# オホーツク海高気圧の見直し

同じ天気図を、**以前の答えを見ずに**判定し直します。

**判定セルは1つだけです。** 下の `MODE` を書き換えて、どの作業をするか選びます。
「すべてを実行」を押しても画面が増えないように、この作りにしてあります。

| MODE | 対象 | 目的 |
|---|---|---|
| `"sample"` | 無作為200枚 | 判断の揺れを測る（済／κ 0.464） |
| `"months"` | 4〜9月 1158枚 | 見落としを拾う（済／κ 0.467） |
| `"candidates"` | 一度でも「あり」とした約205枚 | **基準を1つに決める（いまここ）** |

結果はMODEごとに別のファイルに書かれます。**元の labels.csv は変更しません。**
途中で閉じても、次に開けば続きから再開します。

## 判定基準（始める前に記入してください）

> オホーツク海高気圧を「あり」とするのは、
>
> - オホーツク海（おおむね北緯45〜60度・東経135〜160度）に高気圧の中心があり、
> - かつ ……（ここに条件を書く）
>
> 迷う場合の扱い: ……

1回目と2回目で陽性が 82 → 195 と2.4倍になりました。判断がばらついたのではなく、
**2回目のほうが基準が緩い**という状態です。**ここで決めた基準が最終ラベルになります。**

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "src" / "labels.py").exists()), None)
if repo is None:
    raise SystemExit("リポジトリのルートが見つかりません")
sys.path.insert(0, str(repo))

# 天気図の画像がある場所。環境に合わせて書き換えてください。
IMAGES_DIR = repo.parent / "weather-pattern-classification-data" / "processed"
LABELS_CSV = repo / "data" / "labels.csv"

print("画像:", IMAGES_DIR, "(あり)" if IMAGES_DIR.exists() else "(見つかりません)")
print("ラベル:", LABELS_CSV, "(あり)" if LABELS_CSV.exists() else "(見つかりません)")

In [ ]:
MODE = "candidates"   # "sample" / "months" / "candidates"

import scripts.label_tool as lt

# カーネルが古いコードを掴んだままだと修正が効かない。版が出なければ再起動する。
print("label_tool の版:", getattr(lt, "VERSION", "(古い版です。カーネルを再起動してください)"))
lt.close_review_sessions()

options = {
    "sample":     dict(out="review_okhotsk.csv",       sample=200,  months=None),
    "months":     dict(out="review_okhotsk_full.csv",  sample=None, months=list(range(4, 10))),
    "candidates": dict(out="review_okhotsk_final.csv", sample=None, months=None),
}[MODE]

filenames = None
if MODE == "candidates":
    filenames = lt.positives_union(
        LABELS_CSV,
        [repo / "data" / "review_okhotsk.csv", repo / "data" / "review_okhotsk_full.csv"],
        label="okhotsk_high",
    )
    print(f"一度でも「あり」とした天気図: {len(filenames)}枚")

lt.run_binary_review_session(
    images_dir=IMAGES_DIR,
    labels_csv=LABELS_CSV,
    out_csv=repo / "data" / options["out"],
    label="okhotsk_high",
    sample=options["sample"],
    months=options["months"],
    filenames=filenames,
    image_width=560,
)

## 終わったら

PowerShellで突き合わせます（`MODE` に対応するファイル名にしてください）。

```powershell
python -m scripts.compare_review --review data\review_okhotsk_final.csv --labels data\labels.csv --label okhotsk_high
```

内容を確認したうえで、最終ラベルを別ファイルに書き出します。

```powershell
python -m scripts.compare_review --review data\review_okhotsk_final.csv --labels data\labels.csv --label okhotsk_high --apply data\labels_v2.csv
```

**元の labels.csv は変更されません。** 中身を確認してから差し替えてください。